# Clase 122 — PyTorch: tensores + autograd

Intentamos `import torch`. Si no está, fallback con numpy y autograd manual.

In [ ]:
USE_TORCH = False
try:
    import torch
    USE_TORCH = True
    print('torch:', torch.__version__, '| device cpu')
except Exception as e:
    print('torch no disponible. Fallback numpy + autograd manual. Motivo:', type(e).__name__)
import numpy as np
np.random.seed(42)

## 1. Tensores: creación + operaciones + broadcasting

In [ ]:
if USE_TORCH:
    torch.manual_seed(42)
    a = torch.tensor([[1., 2.], [3., 4.]])
    b = torch.ones(2, 2)
    print('a + b =\n', a + b)
    print('a @ b =\n', a @ b)
    print('broadcast a + [10, 20] =\n', a + torch.tensor([10., 20.]))
    print('device:', a.device)
else:
    a = np.array([[1., 2.], [3., 4.]])
    b = np.ones((2, 2))
    print('a + b =\n', a + b)
    print('a @ b =\n', a @ b)

## 2. Autograd: requires_grad + backward()

In [ ]:
if USE_TORCH:
    x = torch.tensor(3.0, requires_grad=True)
    y = x**2 + 2*x + 1   # (x+1)^2; dy/dx = 2x+2 = 8 en x=3
    y.backward()
    print(f'x={x.item()}, y={y.item()}, dy/dx={x.grad.item()} (esperado 8)')
else:
    # autograd manual con regla de la cadena
    x_val = 3.0
    y_val = x_val**2 + 2*x_val + 1
    dydx = 2*x_val + 2
    print(f'x={x_val}, y={y_val}, dy/dx={dydx} (esperado 8)')

## 3. Gradiente acumulado (importante en RNNs / gradient accumulation)

In [ ]:
if USE_TORCH:
    x = torch.tensor(1.0, requires_grad=True)
    for step in range(3):
        y = x**2
        y.backward()
        print(f'step {step}: x.grad acumulado = {x.grad.item()}')
    x.grad.zero_()
    print('después de zero_grad:', x.grad.item())
else:
    print('numpy: simular acumulación manualmente sumando dy/dx step a step')

## 4. nn.Module mínimo: regresión lineal manual

In [ ]:
# Dataset sintético: y = 3x + 2 + noise
X_np = np.linspace(-2, 2, 100).reshape(-1, 1).astype(np.float32)
y_np = (3*X_np + 2 + np.random.normal(0, 0.3, X_np.shape)).astype(np.float32)

if USE_TORCH:
    import torch.nn as nn
    X = torch.from_numpy(X_np); y = torch.from_numpy(y_np)
    model = nn.Linear(1, 1)
    opt = torch.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.MSELoss()
    for epoch in range(200):
        pred = model(X); loss = loss_fn(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
    w, b = model.weight.item(), model.bias.item()
    print(f'aprendido: w={w:.3f} (esp 3), b={b:.3f} (esp 2)')
else:
    # SGD manual
    w, b = 0.0, 0.0; lr = 0.05
    for epoch in range(200):
        pred = w*X_np + b
        err = pred - y_np
        dw = (2 * err * X_np).mean(); db = (2 * err).mean()
        w -= lr*dw; b -= lr*db
    print(f'aprendido: w={w:.3f} (esp 3), b={b:.3f} (esp 2)')

## 5. Derivada analítica vs autograd

In [ ]:
if USE_TORCH:
    x = torch.linspace(-3, 3, 7, requires_grad=True)
    y = (x**3).sum()   # dy/dx_i = 3 x_i^2
    y.backward()
    print('autograd:', x.grad.numpy())
    print('analytical:', 3 * (x.detach().numpy()**2))
else:
    xs = np.linspace(-3, 3, 7)
    print('analytical 3x^2:', 3 * xs**2)

## Conclusiones

- Tensores PyTorch = numpy + GPU + autograd.
- `.backward()` recorre el grafo dinámico que se construye durante el forward.
- Gradientes se *acumulan* — siempre `zero_grad()` al inicio de cada step.
- `nn.Module` + `optim` ahorran código pero el ciclo manual es importante para custom loops (Clase 121).